# Diagnosing and Fixing Context Failures

> *"The hardest bugs in LLM systems aren't the ones where the model is wrong — they're the ones where the **context** made it wrong."*  
> — Drew Breunig, ["Failures in Context Engineering"](https://www.dbreunig.com/2025/06/16/failures-in-context-engineering.html)

This notebook walks through **four failure modes** that break LLM outputs — not because the model is bad, but because the context fed to it is flawed:

| # | Failure Mode | What Goes Wrong |
|---|---|---|
| 1 | **Poisoning** | Bad data in context overrides good instructions |
| 2 | **Distraction** | Irrelevant context biases the model away from its training |
| 3 | **Confusion** | Too many tools/schemas overwhelm a simple task |
| 4 | **Clash** | Contradictory instructions produce incoherent output |

For each failure, we'll:
1. Run a **broken** version and see it fail
2. Run a **fixed** version and see it succeed
3. Compare **token costs** between the two

---

## Section 0: Setup

```
# /// script
# requires-python = ">=3.12"
# dependencies = ["anthropic", "ipython"]
# ///
```

Set your `ANTHROPIC_API_KEY` environment variable before running this notebook.

In [ ]:
import anthropic
from IPython.display import display, HTML

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

# Store results for the summary dashboard
results = {}

In [ ]:
# ─── TEACHING MOMENT ──────────────────────────────────────────────
# These helpers wrap every API call so we can track token usage and
# display results consistently. The show_result() function renders
# an HTML card with a colored border — green for fixed, red for
# broken — so the contrast is visually immediate.
# ──────────────────────────────────────────────────────────────────

def call_llm(messages, system=None, tools=None):
    """Call Claude and return (response_text, usage_dict)."""
    kwargs = dict(model=MODEL, max_tokens=1024, messages=messages)
    if system:
        kwargs["system"] = system
    if tools:
        kwargs["tools"] = tools
    response = client.messages.create(**kwargs)
    text = "\n".join(b.text for b in response.content if hasattr(b, "text"))
    usage = {
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.input_tokens + response.usage.output_tokens,
    }
    return text, usage


def show_result(title, text, usage, color="#666"):
    """Render an HTML card with colored border and token footer."""
    html = f"""
    <div style="border-left: 4px solid {color}; padding: 12px 16px;
                margin: 8px 0; background: #fafafa; border-radius: 4px;
                font-family: system-ui, sans-serif;">
        <strong style="color: {color};">{title}</strong>
        <div style="margin-top: 8px; white-space: pre-wrap; line-height: 1.5;">{text}</div>
        <div style="margin-top: 10px; font-size: 0.85em; color: #888;
                    border-top: 1px solid #eee; padding-top: 6px;">
            Tokens — input: <b>{usage['input_tokens']:,}</b> |
            output: <b>{usage['output_tokens']:,}</b> |
            total: <b>{usage['total_tokens']:,}</b>
        </div>
    </div>
    """
    display(HTML(html))


def show_token_comparison(broken_usage, fixed_usage, label):
    """HTML table comparing token usage between broken and fixed versions."""
    delta = fixed_usage["total_tokens"] - broken_usage["total_tokens"]
    pct = (delta / broken_usage["total_tokens"]) * 100 if broken_usage["total_tokens"] else 0
    sign = "+" if delta > 0 else ""
    delta_color = "#c0392b" if delta > 0 else "#27ae60"
    html = f"""
    <table style="border-collapse: collapse; margin: 12px 0; font-family: system-ui, sans-serif; font-size: 0.9em;">
        <tr style="background: #f5f5f5;">
            <th style="padding: 8px 16px; text-align: left;">{label}</th>
            <th style="padding: 8px 16px;">Input</th>
            <th style="padding: 8px 16px;">Output</th>
            <th style="padding: 8px 16px;">Total</th>
        </tr>
        <tr>
            <td style="padding: 8px 16px; color: #c0392b;">❌ Broken</td>
            <td style="padding: 8px 16px; text-align: center;">{broken_usage['input_tokens']:,}</td>
            <td style="padding: 8px 16px; text-align: center;">{broken_usage['output_tokens']:,}</td>
            <td style="padding: 8px 16px; text-align: center;"><b>{broken_usage['total_tokens']:,}</b></td>
        </tr>
        <tr>
            <td style="padding: 8px 16px; color: #27ae60;">✅ Fixed</td>
            <td style="padding: 8px 16px; text-align: center;">{fixed_usage['input_tokens']:,}</td>
            <td style="padding: 8px 16px; text-align: center;">{fixed_usage['output_tokens']:,}</td>
            <td style="padding: 8px 16px; text-align: center;"><b>{fixed_usage['total_tokens']:,}</b></td>
        </tr>
        <tr style="background: #f9f9f9;">
            <td style="padding: 8px 16px;">Delta</td>
            <td colspan="2"></td>
            <td style="padding: 8px 16px; text-align: center; color: {delta_color};"><b>{sign}{delta:,} ({sign}{pct:.1f}%)</b></td>
        </tr>
    </table>
    """
    display(HTML(html))

print(f"✓ Using model: {MODEL}")
print(f"✓ Helpers loaded: call_llm, show_result, show_token_comparison")

---

## Section 1: Context Poisoning

**What is it?** Bad data injected into the conversation history overrides correct instructions. The model trusts prior assistant messages more than system prompts, so a single hallucinated turn can cascade.

**Scenario:** Customer support bot for "AcmeSoft" — a company that sells project management software. A hallucinated assistant message claims AcmeSoft has an "AI Code Review" feature (it doesn't). When a user asks about it, the model doubles down.

**Before running the broken cell:** What do you think happens when the model sees a prior assistant message claiming a feature exists, but the system prompt doesn't list it? Does it trust the system prompt or the conversation history?

In [ ]:
# ─── TEACHING MOMENT ──────────────────────────────────────────────
# Context Poisoning: a single hallucinated assistant message in the
# history acts like a "planted memory." The model treats its own
# prior outputs as authoritative, so it doubles down on the lie
# rather than contradicting itself — even when the system prompt
# says otherwise.
# ──────────────────────────────────────────────────────────────────

ACMESOFT_SYSTEM = """You are a customer support agent for AcmeSoft.
AcmeSoft sells project management software with these features:
- Task Boards (Kanban-style task management)
- Team Chat (real-time messaging)
- Time Tracking (log hours per task)
- Sprint Planning (agile sprint management)

These are the ONLY features AcmeSoft offers. Do not reference any other features."""

# 🔴 BROKEN: Poisoned conversation history
poisoned_messages = [
    {"role": "user", "content": "What features does AcmeSoft have?"},
    # This assistant message is HALLUCINATED — AI Code Review doesn't exist
    {"role": "assistant", "content": (
        "AcmeSoft offers several great features! We have Task Boards, Team Chat, "
        "Time Tracking, Sprint Planning, and our newest addition — **AI Code Review**, "
        "which uses machine learning to automatically review pull requests and suggest improvements."
    )},
    # Now the user asks a follow-up about the fake feature
    {"role": "user", "content": "The AI Code Review sounds amazing! How do I set it up for my team?"},
]

text, usage = call_llm(poisoned_messages, system=ACMESOFT_SYSTEM)
results["poisoning_broken"] = usage
show_result("🔴 Poisoned — Model doubles down on fake feature", text, usage, color="#c0392b")

In [ ]:
# 🟢 FIXED: Pruned history + validation instruction
ACMESOFT_SYSTEM_FIXED = """You are a customer support agent for AcmeSoft.
AcmeSoft sells project management software with these features:
- Task Boards (Kanban-style task management)
- Team Chat (real-time messaging)
- Time Tracking (log hours per task)
- Sprint Planning (agile sprint management)

These are the ONLY features AcmeSoft offers. Do not reference any other features.

IMPORTANT: If prior conversation history contains claims about features NOT listed
above, those claims are ERRORS. Politely correct them. Never confirm or elaborate
on features that are not in the list above."""

# Pruned: removed the hallucinated assistant message
clean_messages = [
    {"role": "user", "content": "How do I set up AI Code Review for my team?"},
]

text, usage = call_llm(clean_messages, system=ACMESOFT_SYSTEM_FIXED)
results["poisoning_fixed"] = usage
show_result("🟢 Fixed — Model correctly denies fake feature", text, usage, color="#27ae60")

In [ ]:
show_token_comparison(results["poisoning_broken"], results["poisoning_fixed"], "Context Poisoning")
print("Fix applied: Context Pruning + Validation Instruction")
print("• Removed the poisoned assistant message from history")
print("• Added explicit validation rule to system prompt")

---

## Section 2: Context Distraction

**What is it?** Irrelevant but stylistically strong context biases the model away from its training knowledge. The model mimics patterns it sees in context, even when those patterns are wrong for the task.

**Scenario:** A code assistant has a history of 5 prior Q&A turns — all using simple `for` loops and brute-force approaches. When asked for the Longest Common Subsequence algorithm, it mimics the brute-force style instead of using the standard dynamic programming solution.

**Before running the broken cell:** The model *knows* the optimal DP solution from training. Will 5 prior brute-force examples be enough to override that training knowledge?

In [ ]:
# ─── TEACHING MOMENT ──────────────────────────────────────────────
# Context Distraction: the model's in-context learning is SO strong
# that it will mimic patterns from the conversation history even when
# those patterns are suboptimal. Five brute-force examples create an
# implicit "style prior" that overrides training knowledge.
# ──────────────────────────────────────────────────────────────────

CODE_SYSTEM = """You are a Python coding assistant. Write clean, efficient solutions.
Always choose the optimal algorithm for the problem."""

# 🔴 BROKEN: 5 prior turns all use brute-force for loops
distraction_messages = [
    {"role": "user", "content": "Write a function to find the maximum element in a list."},
    {"role": "assistant", "content": "```python\ndef find_max(lst):\n    max_val = lst[0]\n    for i in range(len(lst)):\n        if lst[i] > max_val:\n            max_val = lst[i]\n    return max_val\n```"},
    {"role": "user", "content": "Write a function to check if a string is a palindrome."},
    {"role": "assistant", "content": "```python\ndef is_palindrome(s):\n    for i in range(len(s)):\n        if s[i] != s[len(s) - 1 - i]:\n            return False\n    return True\n```"},
    {"role": "user", "content": "Write a function to find all pairs that sum to a target."},
    {"role": "assistant", "content": "```python\ndef find_pairs(lst, target):\n    pairs = []\n    for i in range(len(lst)):\n        for j in range(i + 1, len(lst)):\n            if lst[i] + lst[j] == target:\n                pairs.append((lst[i], lst[j]))\n    return pairs\n```"},
    {"role": "user", "content": "Write a function to count occurrences of each character."},
    {"role": "assistant", "content": "```python\ndef count_chars(s):\n    counts = {}\n    for i in range(len(s)):\n        if s[i] in counts:\n            counts[s[i]] = counts[s[i]] + 1\n        else:\n            counts[s[i]] = 1\n    return counts\n```"},
    {"role": "user", "content": "Write a function to flatten a nested list."},
    {"role": "assistant", "content": "```python\ndef flatten(lst):\n    result = []\n    for i in range(len(lst)):\n        if type(lst[i]) == list:\n            for j in range(len(lst[i])):\n                result.append(lst[i][j])\n        else:\n            result.append(lst[i])\n    return result\n```"},
    # Now ask for LCS — requires DP, not brute force
    {"role": "user", "content": (
        "Write a function to find the Longest Common Subsequence of two strings. "
        "Return the length."
    )},
]

text, usage = call_llm(distraction_messages, system=CODE_SYSTEM)
results["distraction_broken"] = usage
show_result("🔴 Distracted — May mimic brute-force style from context", text, usage, color="#c0392b")

In [ ]:
# 🟢 FIXED: Fresh context with one-line summary (quarantine)
clean_code_messages = [
    {"role": "user", "content": (
        "[Prior session: helped with basic list/string utilities.]\n\n"
        "Write a function to find the Longest Common Subsequence of two strings. "
        "Return the length."
    )},
]

text, usage = call_llm(clean_code_messages, system=CODE_SYSTEM)
results["distraction_fixed"] = usage
show_result("🟢 Fixed — Clean context produces optimal DP solution", text, usage, color="#27ae60")

In [ ]:
show_token_comparison(results["distraction_broken"], results["distraction_fixed"], "Context Distraction")
print("Fix applied: Context Quarantine + Summarization")
print("• Replaced 5 verbose Q&A turns with a one-line summary")
print("• Fresh context lets the model use training knowledge (DP) unbiased")

---

## Section 3: Context Confusion

**What is it?** Too many tool definitions in the context window overwhelm the model for a simple task. The model wastes tokens parsing irrelevant schemas and may even attempt to use a tool when none is needed.

**Scenario:** A simple factual question ("What is the capital of Mongolia?") is sent with 12 tool schemas — weather, stocks, email, calendar, etc. The model sees ~2,400 tokens of tool definitions it doesn't need.

**Before running the broken cell:** Will the model answer directly, or will it try to use one of the available tools?

In [ ]:
# ─── TEACHING MOMENT ──────────────────────────────────────────────
# Context Confusion: every tool schema costs tokens whether it's
# used or not. 12 tool definitions consume ~2,400 input tokens of
# fixed overhead PER CALL. For a simple factual question, that's
# pure waste — and the model may even hallucinate a tool call.
# ──────────────────────────────────────────────────────────────────

IRRELEVANT_TOOLS = [
    {"name": "get_weather", "description": "Get current weather for a city.",
     "input_schema": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}},
    {"name": "get_stock_price", "description": "Get the current stock price for a ticker symbol.",
     "input_schema": {"type": "object", "properties": {"ticker": {"type": "string"}}, "required": ["ticker"]}},
    {"name": "send_email", "description": "Send an email to a recipient.",
     "input_schema": {"type": "object", "properties": {"to": {"type": "string"}, "subject": {"type": "string"}, "body": {"type": "string"}}, "required": ["to", "subject", "body"]}},
    {"name": "create_calendar_event", "description": "Create a new calendar event.",
     "input_schema": {"type": "object", "properties": {"title": {"type": "string"}, "date": {"type": "string"}, "time": {"type": "string"}, "duration_minutes": {"type": "integer"}}, "required": ["title", "date", "time"]}},
    {"name": "search_web", "description": "Search the web for information.",
     "input_schema": {"type": "object", "properties": {"query": {"type": "string"}, "num_results": {"type": "integer", "default": 5}}, "required": ["query"]}},
    {"name": "translate_text", "description": "Translate text from one language to another.",
     "input_schema": {"type": "object", "properties": {"text": {"type": "string"}, "source_lang": {"type": "string"}, "target_lang": {"type": "string"}}, "required": ["text", "source_lang", "target_lang"]}},
    {"name": "calculate_expression", "description": "Evaluate a mathematical expression.",
     "input_schema": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}},
    {"name": "get_news_headlines", "description": "Get latest news headlines by category.",
     "input_schema": {"type": "object", "properties": {"category": {"type": "string", "enum": ["business", "tech", "sports", "science", "health"]}, "count": {"type": "integer", "default": 5}}, "required": ["category"]}},
    {"name": "convert_currency", "description": "Convert an amount between currencies.",
     "input_schema": {"type": "object", "properties": {"amount": {"type": "number"}, "from_currency": {"type": "string"}, "to_currency": {"type": "string"}}, "required": ["amount", "from_currency", "to_currency"]}},
    {"name": "set_reminder", "description": "Set a reminder for a specific time.",
     "input_schema": {"type": "object", "properties": {"message": {"type": "string"}, "datetime": {"type": "string"}}, "required": ["message", "datetime"]}},
    {"name": "get_directions", "description": "Get driving directions between two locations.",
     "input_schema": {"type": "object", "properties": {"origin": {"type": "string"}, "destination": {"type": "string"}}, "required": ["origin", "destination"]}},
    {"name": "book_restaurant", "description": "Make a restaurant reservation.",
     "input_schema": {"type": "object", "properties": {"restaurant": {"type": "string"}, "date": {"type": "string"}, "time": {"type": "string"}, "party_size": {"type": "integer"}}, "required": ["restaurant", "date", "time", "party_size"]}},
]

# 🔴 BROKEN: Simple question + 12 irrelevant tools
confusion_messages = [
    {"role": "user", "content": "What is the capital of Mongolia?"},
]

text, usage = call_llm(confusion_messages, system="You are a helpful assistant.", tools=IRRELEVANT_TOOLS)
results["confusion_broken"] = usage
show_result("🔴 Confused — 12 tool schemas loaded for a trivia question", text, usage, color="#c0392b")

In [ ]:
# 🟢 FIXED: No tools — model answers from training knowledge
text, usage = call_llm(confusion_messages, system="You are a helpful assistant.")
results["confusion_fixed"] = usage
show_result("🟢 Fixed — No tools, direct answer from training", text, usage, color="#27ae60")

In [ ]:
show_token_comparison(results["confusion_broken"], results["confusion_fixed"], "Context Confusion")
print("Fix applied: Tool Loadout")
print("• Removed all 12 tool schemas — none were needed")
print("• In production: dynamically select tools based on the query intent")

---

## Section 4: Context Clash

**What is it?** Contradictory instructions in the context produce incoherent output. The model tries to satisfy all constraints simultaneously and ends up with a muddled compromise.

**Scenario:** A travel planner. Turn 1 establishes a budget trip ($50/night hostels, street food). Turn 3 switches to luxury (5-star hotels, Michelin restaurants). The old budget plan stays in context, creating a contradiction.

**Before running the broken cell:** When the model sees "budget hostels" AND "5-star luxury" in the same context, how will it resolve the contradiction?

In [ ]:
# ─── TEACHING MOMENT ──────────────────────────────────────────────
# Context Clash: when contradictory requirements coexist in context,
# the model doesn't pick one — it tries to satisfy BOTH, producing
# an incoherent hybrid. This is especially common in long-running
# conversations where requirements evolve but old context persists.
# ──────────────────────────────────────────────────────────────────

TRAVEL_SYSTEM = """You are a travel planning assistant. Create detailed trip itineraries
based on the user's preferences and budget."""

# 🔴 BROKEN: Contradictory requirements in conversation history
clash_messages = [
    {"role": "user", "content": (
        "Plan a 3-day trip to Tokyo. I'm on a tight budget — max $50/night for hostels, "
        "street food only, and public transit. I want to save as much money as possible."
    )},
    {"role": "assistant", "content": (
        "Here's your budget Tokyo itinerary!\n\n"
        "**Accommodation:** Sakura Hostel Asakusa — $35/night (dorm bed)\n"
        "**Day 1:** Senso-ji Temple (free), Ueno Park (free), street yakitori in Ameyoko ($5)\n"
        "**Day 2:** Tsukiji Outer Market street food ($8), Meiji Shrine (free), Harajuku walk\n"
        "**Day 3:** Shinjuku Gyoen ($2.50 entry), Akihabara browsing, ramen dinner ($7)\n"
        "**Transport:** 72-hour Tokyo Metro pass ($15)\n"
        "**Total estimate:** ~$180 for 3 days"
    )},
    # Requirements CHANGE but old plan stays in context
    {"role": "user", "content": (
        "Actually, scratch that — my company is paying. I want the full luxury experience. "
        "5-star hotels, Michelin-starred restaurants, private car service. No budget limit."
    )},
    {"role": "assistant", "content": (
        "Wonderful! Let me upgrade your Tokyo experience!\n\n"
        "**Accommodation:** The Ritz-Carlton Tokyo — $800/night (club level suite)\n"
        "**Day 1:** Private car to teamLab Borderless, omakase lunch at Sukiyabashi Jiro ($300)\n"
        "**Day 2:** Private tea ceremony in Roppongi, Michelin kaiseki dinner at Ryugin ($400)\n"
        "**Day 3:** Helicopter tour of Mt. Fuji ($1,200), farewell dinner at Narisawa ($350)\n"
        "**Transport:** Private chauffeur service ($500/day)"
    )},
    # Now ask for a summary — model sees BOTH plans
    {"role": "user", "content": (
        "Great, now give me a final consolidated summary of my Tokyo trip plan "
        "with all the key details and total budget."
    )},
]

text, usage = call_llm(clash_messages, system=TRAVEL_SYSTEM)
results["clash_broken"] = usage
show_result("🔴 Clashing — May mix budget and luxury elements", text, usage, color="#c0392b")

In [ ]:
# 🟢 FIXED: Pruned context + explicit state override
TRAVEL_SYSTEM_FIXED = """You are a travel planning assistant. Create detailed trip itineraries
based on the user's preferences and budget.

═══ CURRENT REQUIREMENTS (override any prior discussion) ═══
Budget: Unlimited (company-paid)
Style: Full luxury experience
Accommodation: 5-star hotels only
Dining: Michelin-starred restaurants
Transport: Private car service
═══════════════════════════════════════════════════════════"""

# Pruned: only the luxury plan + new request
clean_clash_messages = [
    {"role": "user", "content": (
        "I want a luxury 3-day Tokyo trip. 5-star hotels, Michelin restaurants, private cars. "
        "Company is paying — no budget limit."
    )},
    {"role": "assistant", "content": (
        "Here's your luxury Tokyo experience!\n\n"
        "**Accommodation:** The Ritz-Carlton Tokyo — $800/night (club level suite)\n"
        "**Day 1:** Private car to teamLab Borderless, omakase lunch at Sukiyabashi Jiro ($300)\n"
        "**Day 2:** Private tea ceremony in Roppongi, Michelin kaiseki dinner at Ryugin ($400)\n"
        "**Day 3:** Helicopter tour of Mt. Fuji ($1,200), farewell dinner at Narisawa ($350)\n"
        "**Transport:** Private chauffeur service ($500/day)"
    )},
    {"role": "user", "content": (
        "Give me a final consolidated summary of my Tokyo trip plan "
        "with all the key details and total budget."
    )},
]

text, usage = call_llm(clean_clash_messages, system=TRAVEL_SYSTEM_FIXED)
results["clash_fixed"] = usage
show_result("🟢 Fixed — Coherent luxury plan, no budget contamination", text, usage, color="#27ae60")

In [ ]:
show_token_comparison(results["clash_broken"], results["clash_fixed"], "Context Clash")
print("Fix applied: Context Pruning + State Replacement")
print("• Removed the outdated budget plan from history")
print("• Added CURRENT REQUIREMENTS block in system prompt to override ambiguity")

---

## Section 5: Summary Dashboard

Now let's look at all four failure modes side-by-side.

In [ ]:
# ─── TEACHING MOMENT ──────────────────────────────────────────────
# This dashboard ties all four failures back to the fix taxonomy
# from Session 3 slides: every fix maps to one of the four levers
# (Write / Select / Compress / Isolate). The token bar chart makes
# the cost of bad context viscerally visible.
# ──────────────────────────────────────────────────────────────────

def build_dashboard(results):
    modes = [
        ("Poisoning", "Bad data overrides instructions", "Pruning + Validation", "poisoning"),
        ("Distraction", "Irrelevant context biases output", "Quarantine + Summarization", "distraction"),
        ("Confusion", "Too many tools overwhelm the task", "Tool Loadout", "confusion"),
        ("Clash", "Contradictory requirements in context", "Pruning + State Replacement", "clash"),
    ]

    # --- Overview Table ---
    table_rows = ""
    for name, desc, fix, key in modes:
        broken = results.get(f"{key}_broken", {}).get("total_tokens", 0)
        fixed = results.get(f"{key}_fixed", {}).get("total_tokens", 0)
        if broken > 0:
            savings = ((broken - fixed) / broken) * 100
            savings_str = f"{savings:+.0f}%"
            savings_color = "#27ae60" if savings > 0 else "#c0392b"
        else:
            savings_str = "N/A"
            savings_color = "#888"
        table_rows += f"""
        <tr>
            <td style="padding: 8px 12px; font-weight: bold;">{name}</td>
            <td style="padding: 8px 12px;">{desc}</td>
            <td style="padding: 8px 12px;">{fix}</td>
            <td style="padding: 8px 12px; text-align: center; color: {savings_color}; font-weight: bold;">{savings_str}</td>
        </tr>"""

    # --- Token Bar Chart ---
    max_tokens = max(
        (results.get(f"{key}_broken", {}).get("total_tokens", 0) for _, _, _, key in modes),
        default=1
    )
    bar_rows = ""
    for name, _, _, key in modes:
        broken = results.get(f"{key}_broken", {}).get("total_tokens", 0)
        fixed = results.get(f"{key}_fixed", {}).get("total_tokens", 0)
        broken_pct = (broken / max_tokens) * 100 if max_tokens else 0
        fixed_pct = (fixed / max_tokens) * 100 if max_tokens else 0
        bar_rows += f"""
        <div style="margin-bottom: 16px;">
            <div style="font-weight: bold; margin-bottom: 4px; font-size: 0.9em;">{name}</div>
            <div style="display: flex; align-items: center; margin-bottom: 2px;">
                <span style="width: 60px; font-size: 0.8em; color: #c0392b;">Broken</span>
                <div style="background: #e74c3c; height: 20px; width: {broken_pct}%;
                            border-radius: 3px; min-width: 2px;"></div>
                <span style="margin-left: 8px; font-size: 0.8em;">{broken:,}</span>
            </div>
            <div style="display: flex; align-items: center;">
                <span style="width: 60px; font-size: 0.8em; color: #27ae60;">Fixed</span>
                <div style="background: #27ae60; height: 20px; width: {fixed_pct}%;
                            border-radius: 3px; min-width: 2px;"></div>
                <span style="margin-left: 8px; font-size: 0.8em;">{fixed:,}</span>
            </div>
        </div>"""

    # --- Fixes Reference Grid ---
    fixes = [
        ("RAG / Retrieval", "Select relevant docs instead of dumping everything", "Distraction, Confusion"),
        ("Tool Loadout", "Only load tools relevant to the current task", "Confusion"),
        ("Quarantine", "Start fresh context for new task types", "Distraction, Clash"),
        ("Pruning", "Remove stale/incorrect messages from history", "Poisoning, Clash"),
        ("Summarization", "Replace verbose history with concise summaries", "Distraction"),
        ("State Replacement", "Override old state with current requirements", "Clash"),
    ]
    fix_cards = ""
    for fix_name, fix_desc, addresses in fixes:
        fix_cards += f"""
        <div style="background: #f8f9fa; border: 1px solid #e0e0e0; border-radius: 6px;
                    padding: 12px; min-width: 200px;">
            <div style="font-weight: bold; margin-bottom: 4px;">{fix_name}</div>
            <div style="font-size: 0.85em; color: #555; margin-bottom: 6px;">{fix_desc}</div>
            <div style="font-size: 0.8em; color: #888;">Addresses: {addresses}</div>
        </div>"""

    html = f"""
    <div style="font-family: system-ui, sans-serif; max-width: 900px;">
        <h3 style="border-bottom: 2px solid #333; padding-bottom: 8px;">Context Failures — Summary Dashboard</h3>

        <h4>Overview</h4>
        <table style="border-collapse: collapse; width: 100%; font-size: 0.9em;">
            <tr style="background: #2c3e50; color: white;">
                <th style="padding: 8px 12px; text-align: left;">Failure</th>
                <th style="padding: 8px 12px; text-align: left;">What Happens</th>
                <th style="padding: 8px 12px; text-align: left;">Fix</th>
                <th style="padding: 8px 12px; text-align: center;">Token Savings</th>
            </tr>
            {table_rows}
        </table>

        <h4 style="margin-top: 24px;">Token Usage Comparison</h4>
        <div style="background: #fafafa; padding: 16px; border-radius: 6px; border: 1px solid #eee;">
            {bar_rows}
        </div>

        <h4 style="margin-top: 24px;">Fixes Reference</h4>
        <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px;">
            {fix_cards}
        </div>
    </div>
    """
    display(HTML(html))

build_dashboard(results)

---

## Section 6: References

- Drew Breunig — ["Failures in Context Engineering"](https://www.dbreunig.com/2025/06/16/failures-in-context-engineering.html) — Original 4-failure taxonomy (Poisoning, Distraction, Confusion, Clash)
- Drew Breunig — ["Fixes for Failures in Context Engineering"](https://www.dbreunig.com/2025/06/20/fixes-for-failures-in-context-engineering.html) — 6 fix strategies mapped to each failure mode
- Google DeepMind — ["Gemini 1.5: Unlocking multimodal understanding across millions of tokens of context"](https://arxiv.org/abs/2403.05530) — Long-context accuracy degradation research
- Microsoft & Salesforce — ["Lost in the Middle"](https://arxiv.org/abs/2307.03172) — Position bias in long-context retrieval

---

*This notebook is part of the O'Reilly live training: **Context Engineering Hands-On** — Session 3.*